In [ ]:
import matplotlib as mpl
from matplotlib import pyplot as plt
import numpy as np
from scipy.fft import fft, fftfreq, ifft
import math
import soundfile as sf

In [ ]:
#data_oboe_c5, samplerate_oboe_c5 = sf.read('../data/Oboe.ff.C5.stereo.aif')
#data_oboe_e5, samplerate_oboe_e5 = sf.read('../data/Oboe.ff.E5.stereo.aif')
#data_oboe_g5, samplerate_oboe_g5 = sf.read('../data/Oboe.ff.G5.stereo.aif')
#data_oboe_c4, samplerate_oboe_c4 = sf.read('../data/Oboe.ff.C4.stereo.aif')
#data_oboe_c6, samplerate_oboe_c6 = sf.read('../data/Oboe.ff.C6.stereo.aif')

notes = ["C5", "Db5", "D5", "Eb5", "E5", "F5", "Gb5", "G5", "Ab5", "A5", "Bb5", "B5"]
data_flute_nonvib = []
samplerate_flute_nonvib = []
data_oboe = []
samplerate_oboe = []

for n in notes:
    data, samplerate = sf.read(f'../data/Flute.nonvib.ff.{n}.stereo.aif')
    data_flute_nonvib.append(data)
    samplerate_flute_nonvib.append(samplerate)

for n in notes:
    data, samplerate = sf.read(f'../data/Oboe.ff.{n}.stereo.aif')
    data_oboe.append(data)
    samplerate_oboe.append(samplerate)

#data_flute_ab5, samplerate_flute_ab5 = sf.read('../data/Flute.nonvib.ff.Ab5.stereo.aif')
#data_flute_a5, samplerate_flute_a5 = sf.read('../data/Flute.nonvib.ff.A5.stereo.aif')
#data_flute_bb5, samplerate_flute_bb5 = sf.read('../data/Flute.nonvib.ff.Bb5.stereo.aif')
#data_flute_b5, samplerate_flute_b5 = sf.read('../data/Flute.nonvib.ff.B5.stereo.aif')
#data_flute_c5, samplerate_flute_c5 = sf.read('../data/Flute.nonvib.ff.C5.stereo.aif')
#data_flute_db5, samplerate_flute_db5 = sf.read('../data/Flute.nonvib.ff.Db5.stereo.aif')
#data_flute_d5, samplerate_flute_d5 = sf.read('../data/Flute.nonvib.ff.D5.stereo.aif')
#data_flute_eb5, samplerate_flute_eb5 = sf.read('../data/Flute.nonvib.ff.Eb5.stereo.aif')
#data_flute_e5, samplerate_flute_e5 = sf.read('../data/Flute.nonvib.ff.E5.stereo.aif')
#data_flute_f5, samplerate_flute_f5 = sf.read('../data/Flute.nonvib.ff.F5.stereo.aif')
#data_flute_gb5, samplerate_flute_gb5 = sf.read('../data/Flute.nonvib.ff.Gb5.stereo.aif')
#data_flute_g5, samplerate_flute_g5 = sf.read('../data/Flute.nonvib.ff.G5.stereo.aif')

In [ ]:
shortest_time = min(data_oboe_c5.shape[0], data_oboe_e5.shape[0], data_oboe_g5.shape[0])
print(shortest_time)

In [ ]:
plt.plot(data_oboe_c5[:, 0], 'b-', data_oboe_e5[:,0], 'r-')
#plt.xlim(0, 10000)
plt.xlim(4000, 5000)
#plt.ylim(-1e-5, 1e-5)
plt.show()

In [ ]:
#data_chord = (data_oboe_c5[:shortest_time] + data_oboe_e5[:shortest_time] + data_oboe_g5[:shortest_time]) / 3
data_chord = (data_oboe_c5[:shortest_time] + data_oboe_e5[:shortest_time]) / 2

In [ ]:
sf.write("c_major.wav", data_chord, samplerate_c5)

In [ ]:
data_oboe_c4, samplerate_oboe_c4 = sf.read('../data/Oboe.ff.C4.stereo.aif')
data_oboe_c5, samplerate_oboe_c5 = sf.read('../data/Oboe.ff.C5.stereo.aif')
data_oboe_c6, samplerate_oboe_c6 = sf.read('../data/Oboe.ff.C6.stereo.aif')
shortest_time = min(data_oboe_c4.shape[0], data_oboe_c5.shape[0], data_oboe_c6.shape[0])
data_octaves = (data_oboe_c4[:shortest_time] + data_oboe_c5[:shortest_time] + data_oboe_c6[:shortest_time]) / 3
sf.write("c_octaves.wav", data_octaves, samplerate_oboe_c5)

# Try FFT transforms

In [ ]:
def freq_to_note(frequency):
    """
    Convert a frequency in Hz to the closest musical note.
    
    Returns a tuple: (note_name, octave, exact_frequency_of_that_note, cents_off)
    """
    if frequency <= 0:
        raise ValueError("Frequency must be positive")

    note_names = ['C', 'Db', 'D', 'Eb', 'E', 'F', 'Gb', 'G', 'Ab', 'A', 'Bb', 'B']
    
    A4 = 440.0
    
    # Number of semitones away from A4 (can be fractional)
    semitones_from_a4 = 12 * math.log2(frequency / A4)
    
    # Round to nearest semitone to find the closest note
    nearest_semitone = round(semitones_from_a4)
    
    # How far off (in cents) the input frequency is from that exact note
    #cents_off = round((semitones_from_a4 - nearest_semitone) * 100, 1)
    
    # MIDI note number: A4 = 69
    midi_number = 69 + nearest_semitone
    
    # Octave and note index (MIDI: C-1 = 0, so octave = midi//12 - 1)
    octave = midi_number // 12 - 1
    note_index = midi_number % 12
    note_name = note_names[note_index]
    
    # Exact frequency of the identified note
    #exact_freq = A4 * (2 ** (nearest_semitone / 12))

    return f"{note_name}{octave}"
    #return note_name, octave, round(exact_freq, 2)

In [ ]:
def heuristic_dominant_harmonic(X, freqs):

    max_freq_idx = np.argmax(np.abs(X))
    max_freq = freqs[max_freq_idx]

    print((max_freq, np.abs(X[max_freq_idx])))
    print(np.abs(X[int(0.5*max_freq_idx)]))
    print(np.abs(X[2*max_freq_idx]))
    
    # Check 2x and 0.5x the dominant frequecy. If 2x is stronger than 0.5, return 2x.
    # Otherwise, return 1x
    if np.abs(X[int(2*max_freq_idx)]) > np.abs(X[int(0.5*max_freq_idx)]):
        return freqs[2*max_freq_idx]
    else:
        return freqs[max_freq_idx]
    

def plot_freq_space(data, samplerate):
    # 1. Create a sample signal: two sine waves + noise
    #fs = 500              # sampling rate (samples per second)
    fs = samplerate
    #T = 1.0               # duration in seconds
    #N = int(fs * T)       # number of samples
    N = data.shape[0]
    T = N / fs
    t = np.linspace(0, T, N, endpoint=False)
    
    #signal = (np.sin(2 * np.pi * 50 * t)      # 50 Hz component
    #          + 0.5 * np.sin(2 * np.pi * 120 * t)  # 120 Hz component
    #          + 0.2 * np.random.randn(N))     # noise
    signal = data
    
    # 2. Compute the DFT
    X = fft(signal)
    freqs = fftfreq(N, d=1/fs)   # corresponding frequency bins
    
    # 3. Take the inverse to recover the original signal
    recovered = ifft(X)
    
    # 4. Plot: time domain, frequency domain, and reconstruction check
    #fig, axs = plt.subplots(2, 1, figsize=(8, 8))
    #fig, axs = plt.subplots(1, 1, figsize=(8, 8))
    fig, axs = plt.subplots(2, 1, figsize=(8, 4))
    
    axs[0].plot(t, signal)
    axs[0].set_title("Time domain (original signal)")
    axs[0].set_xlabel("Time [s]")
    
    # only plot the positive-frequency half; it's symmetric for real input
    half = N // 2
    axs[1].stem(freqs[:half], np.abs(X[:half]))
    axs[1].set_title("Frequency domain (magnitude spectrum)")
    axs[1].set_xlabel("Frequency [Hz]")
    
    #axs[2].plot(t, recovered.real)
    #axs[2].set_title("Recovered signal (inverse FFT)")
    #axs[2].set_xlabel("Time [s]")
    
    plt.tight_layout()
    plt.show()

    max_freq_idx = np.argmax(np.abs(X[:half]))
    max_freq = freqs[max_freq_idx]
    #my_guess = heuristic_dominant_harmonic(X[:half], freqs[:half])
    #print(f"Strongest frequency is {max_freq}, and divided by 2 is {max_freq/2}. My guess is {my_guess}.")
    print(f"Strongest frequency is {max_freq}, and divided by 2 is {max_freq/2}.")
    print(f"Candidate notes are {freq_to_note(max_freq)} and {freq_to_note(max_freq/2)}")
    
    

In [ ]:
plot_freq_space(data_oboe[0][4000:5000, 0], samplerate_oboe[0])

In [ ]:
for data, samplerate in zip(data_oboe, samplerate_oboe):
    plot_freq_space(data[5000:6000, 0], samplerate)

In [ ]:
for data, samplerate in zip(data_flute_nonvib, samplerate_flute_nonvib):
    plot_freq_space(data[5000:6000, 0], samplerate)